In [1]:
import ibm_db
import ibm_db_dbi
import configparser
import pandas as pd

config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\DB2STAT_connect.ini')

conn_str=config['DB2STAT']['conn_str']
ibm_db_conn = ibm_db.connect(conn_str,'','')
print("SUCCESS")

SUCCESS


In [33]:
# Fetch data using ibm_db_dbi
QUERY="""SELECT  
        P.POI_ID,
        coalesce(L.LOC_ID,0) AS CITY_ID,
        P.POI_LAT* 0.000001 AS CITY_CENTER_LATITUDE,
        P.POI_LAT,
        P.POI_LON* 0.000001 AS CITY_CENTER_LONGITUDE,
        P.POI_LON,
        MAX(P.CTS) AS LATEST_TIME,
        HD.DESC_STRING AS CITY_NAME
FROM HRSDB.HTL_POI_ALL_DA P
LEFT JOIN HRSDB.HTL_LOC_POI_ALL_LU L ON L.POI_ID=P.POI_ID
LEFT JOIN HRSDB.HTL_DESCRIPTION_ALL_DA HD ON HD.DESC_ID=P.POI_ID
WHERE POIGR_ID IN (70)
AND HD.DESC_ID_TYPE=5
AND DESC_DEFAULT=1
GROUP BY 
P.POI_ID,
L.LOC_ID,
P.POI_LAT,
P.POI_LON,
HD.DESC_STRING"""

conn = ibm_db_dbi.Connection(ibm_db_conn)
df = pd.read_sql(QUERY, conn)
print(df.head())

   POI_ID  CITY_ID  CITY_CENTER_LATITUDE   POI_LAT  CITY_CENTER_LONGITUDE  \
0     234      268             47.863644  47863644              12.010200   
1     527        0             51.461700  51461700               6.965530   
2     528        0             50.839900  50839900              12.883000   
3     651      799             46.768970  46768970               8.671789   
4     955        0             47.284900  47284900              11.432400   

    POI_LON                LATEST_TIME                   CITY_NAME  
0  12010200 2016-03-07 14:17:01.781347  Aibling, Bad (bei München)  
1   6965530 2005-05-30 08:33:57.294544       Altendorf (bei Essen)  
2  12883000 2005-05-30 08:33:57.339626    Altendorf (bei Chemnitz)  
3   8671789 2008-03-11 17:21:41.829539        Silenen-Amsteg (Uri)  
4  11432400 2006-03-28 13:04:46.857266          Arzl bei Innsbruck  


In [93]:
import re

# CXL_days
df['CXL_days'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=P)(.*)(?=D)', x)).str[0].astype(float)
df['CXL_days'] = df['CXL_days'].fillna(0)

# CXL_hours
df['CXL_hours'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=T)(.*)(?=H)', x.replace('-', ''))).str[0].astype(float)
df['CXL_hours'] = df['CXL_hours'].fillna(0)

# CXL_minutes
df['CXL_minutes'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=H)(.*)(?=M)', x.replace('-', ''))).str[0].astype(float)
df['CXL_minutes'] = df['CXL_minutes'].fillna(0)

# CXL_seconds
df['CXL_seconds'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=M)(.*)(?=S)', x.replace('-', ''))).str[0].astype(float)
df['CXL_seconds'] = df['CXL_seconds'].fillna(0)

# Total_hours
df['Total_hours'] = ((df.CXL_days*24) + df.CXL_hours)

# Total_hours_6
df['Total_hours_6'] = (((df.CXL_days*24) + df.CXL_hours) - 6)
df['Total_hours_6'] = df['Total_hours_6'].apply(lambda x: 0 if x <0  else x)

df.head()

,HOTEL_ID_VALUE,COMPANY_ID_VALUE,DATE_RANGE_FROM_DATE,DATE_RANGE_TO_DATE,CNCLLTN_POLICY_RELATIV_DEADLIN,CXL_days,CXL_hours,CXL_minutes,CXL_seconds,Total_hours,Total_hours_6
0,1,181.0,2018-01-01,2018-02-28,PT0S,0.0,0.0,0.0,0.0,0.0,0.0
1,1,181.0,2018-02-06,2018-02-09,P7D,7.0,0.0,0.0,0.0,168.0,162.0
2,1,6594.0,2018-09-14,2018-09-15,P7DT-5H-59M-59S,7.0,5.0,59.0,59.0,173.0,167.0
3,1,15520.0,2018-01-01,2018-12-31,PT0S,0.0,0.0,0.0,0.0,0.0,0.0
4,1,15536.0,2018-02-06,2018-02-09,PT0S,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
df.dtypes

POI_ID                            int64
CITY_ID                          object
CITY_CENTER_LATITUDE            float64
POI_LAT                           int64
CITY_CENTER_LONGITUDE           float64
POI_LON                           int64
LATEST_TIME              datetime64[ns]
CITY_NAME                        object
dtype: object

In [ ]:
df['CITY_ID'].apply(int)

In [4]:
df.to_csv('C:\\Users\\svi02\\Documents\\db2_city_center_geocodes.csv', 
          sep = '|', 
          header = True,
          encoding='utf-8',
          index=False)

In [29]:
#df.CITY_ID = df.CITY_ID.astype(int)
#df['CITY_ID'].astype(str).astype(int)
df["CITY_ID"]=df["CITY_ID"].astype(int)

ValueError: invalid literal for int() with base 10: 'None'

In [23]:
df.CITY_ID =df.CITY_ID.fillna(0, inplace=True)

In [31]:
df.head()

,POI_ID,CITY_ID,CITY_CENTER_LATITUDE,POI_LAT,CITY_CENTER_LONGITUDE,POI_LON,LATEST_TIME,CITY_NAME
0,81,None,48.937342,48937342,12.040719,12040719,2008-03-20 11:01:16.423152,"Abbach, Bad"
1,815,None,9.586700,9586700,-69.890400,-69890400,2005-05-30 08:34:10.636948,Anzoategui
2,1303,None,51.163153,51163153,13.730379,13730379,2016-03-07 14:17:01.781347,Radeburg - Bärnsdorf (Sachsen)
3,1382,None,13.154240,13154240,-59.549700,-59549700,2006-09-26 17:19:12.599201,Barbados
4,1487,None,53.724050,53724050,12.918330,12918330,2008-03-20 11:01:16.884191,Stavenhagen - Basepohl (Mecklenburg-Vorpommern)
